# 03 — Baseline Model (Phase 4)

Transfer learning on a **frozen ImageNet-pretrained ResNet18** backbone with a new 2-class head — chosen over a from-scratch CNN as a deliberate scope choice, and freezing the backbone is a deliberate choice for a 535-image training set (fine-tuning the whole network would risk overfitting fast). Class-weighted `CrossEntropyLoss` handles the 73%/27% imbalance.

In [ ]:
import sys, time
sys.path.insert(0, "..")
import torch
from torch.utils.data import DataLoader

from src.data_utils import load_dataset_metadata, train_val_test_split
from src.train import HYGDDataset, build_model, class_weights, train

torch.manual_seed(42)
df = load_dataset_metadata("../data/raw")
train_df, val_df, test_df = train_val_test_split(df)

train_loader = DataLoader(HYGDDataset(train_df), batch_size=32, shuffle=True)
val_loader = DataLoader(HYGDDataset(val_df), batch_size=32, shuffle=False)

model = build_model(num_classes=2, freeze_backbone=True)
weights = class_weights(train_df["label"].values)
print("class weights [GON-, GON+]:", weights)

In [ ]:
model, history = train(model, train_loader, val_loader, epochs=8, lr=1e-3, device="cpu", weights=weights)

**Verified run (2026-07-05, CPU, ~5 minutes for 8 epochs):**

```
epoch 1/8  train_loss=0.5210  val_loss=0.3933
epoch 2/8  train_loss=0.3311  val_loss=0.2797
epoch 3/8  train_loss=0.2825  val_loss=0.2995
epoch 4/8  train_loss=0.2251  val_loss=0.2222
epoch 5/8  train_loss=0.2219  val_loss=0.1951
epoch 6/8  train_loss=0.1959  val_loss=0.2063
epoch 7/8  train_loss=0.1851  val_loss=0.1861
epoch 8/8  train_loss=0.1788  val_loss=0.1889
```

![loss curves](../figures/07_loss_curves.png)

Loss decreases smoothly with no signs of divergence; val loss tracks train loss closely (no dramatic overfitting within 8 epochs of head-only training).

In [ ]:
import os
os.makedirs("../results", exist_ok=True)
torch.save(model.state_dict(), "../results/baseline_resnet18.pt")
import json
with open("../results/history.json", "w") as f:
    json.dump(history, f, indent=2)

## Test-set evaluation

In [ ]:
from torch.utils.data import DataLoader
from src.evaluate import evaluate, roc_points

test_loader = DataLoader(HYGDDataset(test_df), batch_size=32, shuffle=False)
metrics = evaluate(model, test_loader, device="cpu")
print("AUC:", metrics["auc"])
print("Sensitivity:", metrics["sensitivity"])
print("Specificity:", metrics["specificity"])
print("Confusion matrix:\n", metrics["confusion_matrix"])

**Verified test-set result (99 images / 44 held-out patients):**

| Metric | Value |
|---|---|
| AUC | **0.952** |
| Sensitivity | 0.892 |
| Specificity | 0.971 |

Confusion matrix `[[TN=33, FP=1], [FN=7, TP=58]]`.

![ROC curve](../figures/05_roc_curve.png)
![confusion matrix](../figures/06_confusion_matrix.png)

A strong baseline result for a frozen-backbone, head-only model on 535 training images. Read this with real caution though — see the README Limitations section (single dataset, single-hospital source, no external validation, small test set of 44 patients).